# TK 1 Analisis Numerik — Soal 2 (SETAR)
## Tugas (i), (ii), dan (iii)

| Tugas | Isi |
|---|---|
| (i) | Membentuk sistem persamaan linear *overdetermined* $A\vec{x} = \vec{b}$ |
| (ii) | Memeriksa potensi masalah numerik pada matriks desain $A$ |
| (iii) | Menyelesaikan $A^{T}A\vec{x} = A^{T}\vec{b}$ dan membandingkan $\kappa_2(A^{T}A)$ dengan $\kappa_2(A)$ |

In [2]:
import csv
from math import sqrt, log10

DATA = "data/stock_train.csv"

---
# Tugas (i) — Formulasi Sistem Overdetermined

### Model SETAR

$$R_t = \begin{cases}
\alpha_1 + \phi_{1,1} R_{t-1} + \phi_{1,2} R_{t-2} + \varepsilon_t, & R_{t-1} \geq 0 \quad \text{(Rezim 1, Bullish)}\\
\alpha_2 + \phi_{2,1} R_{t-1} + \phi_{2,2} R_{t-2} + \varepsilon_t, & R_{t-1} < 0 \quad \text{(Rezim 2, Bearish)}
\end{cases}$$

### Langkah 1 — Hitung deret return

$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

In [3]:
def baca_harga(path):
    harga = []
    with open(path, newline="", encoding="utf-8-sig") as f:
        for baris in csv.DictReader(f):
            harga.append(float(baris["Close"]))
    return harga


def hitung_return(harga):
    return [(harga[i] - harga[i - 1]) / harga[i - 1]
            for i in range(1, len(harga))]


harga = baca_harga(DATA)
R = hitung_return(harga)

print(f"Deret return: {R}")

Deret return: [0.00243799999999992, -0.00831971653109715, 0.007377542254385397, -0.01309624617547668, 0.003582858536279337, 0.0009517491415102273, 0.00913072294374907, 0.004406772555638586, 0.0072544276852810645, 0.006704933591954173, 0.010928303040637901, 0.0034240433376085118, 0.01322917200049082, 0.0017229970040638421, 0.0075121768274284, 0.006852453533600342, 0.021097751162140417, 0.003361545318372704, 0.009377678944322034, 0.003269251883018204, -0.00334090025233117, 0.005784957477159627, -0.00675766507784845, -0.00458541840580404, 0.0016471487900071701, 0.008424453235037523, -0.04357001856561435, 0.01670038860519632, -0.0012393876273065086, -0.006574859256337677, 0.012994224372996004, 0.011836611318222383, 0.0030796131618518172, 0.008143979584494934, 0.010190337067080162, 0.0011634655713456778, 0.0012791287632468171, -0.007129704962959798, 0.005052329300814371, 0.007573922177770766, 0.0051665990859299914, 0.001074262713285732, -0.0030094692911567644, -0.0032051678910101217, 0.0040

### Langkah 2 — Ubah model menjadi linear

Percabangan pada model ditentukan oleh tanda $R_{t-1}$, yang sudah
diketahui dari data. Jadi dengan indikator $I_t = 1$ bila
$R_{t-1} \geq 0$ dan $I_t = 0$ bila sebaliknya, kedua cabang bisa
digabung jadi satu baris:

$$A_{t,:} = \begin{bmatrix} I_t & I_t R_{t-1} & I_t R_{t-2} & 1-I_t & (1-I_t)R_{t-1} & (1-I_t)R_{t-2}\end{bmatrix}, \qquad b_t = R_t$$

$$\vec{x} = [\alpha_1,\ \phi_{1,1},\ \phi_{1,2},\ \alpha_2,\ \phi_{2,1},\ \phi_{2,2}]^{T}$$

In [4]:
NAMA_PARAM = ["alpha_1", "phi_1,1", "phi_1,2",
              "alpha_2", "phi_2,1", "phi_2,2"]
NAMA_KOLOM = ["I", "I*R(t-1)", "I*R(t-2)",
              "1-I", "(1-I)*R(t-1)", "(1-I)*R(t-2)"]
LAG = 2
THRESHOLD = 0.0


def bentuk_sistem(R):
    A, b, rezim = [], [], []
    for t in range(LAG, len(R)):
        r1, r2 = R[t - 1], R[t - 2]
        if r1 >= THRESHOLD:
            # Rezim 1: Bullish
            A.append([1.0, r1, r2, 0.0, 0.0, 0.0])
            rezim.append(1)
        else:
            # Rezim 2: Bearish
            A.append([0.0, 0.0, 0.0, 1.0, r1, r2])
            rezim.append(2)
        b.append(R[t])
    return A, b, rezim


A, b, rezim = bentuk_sistem(R)

### Langkah 3 — Dimensi sistem

In [5]:
m, n = len(A), len(A[0])

print(f"A : {m} x {n}")
print(f"x : {n} x 1")
print(f"b : {m} x 1")
print()
print(f"Rezim 1 (Bullish) : {rezim.count(1)} baris")
print(f"Rezim 2 (Bearish) : {rezim.count(2)} baris")

A : 300 x 6
x : 6 x 1
b : 300 x 1

Rezim 1 (Bullish) : 195 baris
Rezim 2 (Bearish) : 105 baris


Pemeriksaan memakai contoh 6 harga pada dokumen tugas:

In [6]:
A_uji, b_uji, rezim_uji = bentuk_sistem(
    hitung_return([10000.0, 10100.0, 9900.0, 10050.0, 10000.0, 10200.0]))

print("Rezim :", rezim_uji, " (dokumen: Bearish, Bullish, Bearish)")
for baris, bb in zip(A_uji, b_uji):
    print("  ".join(f"{v:8.4f}" for v in baris) + f"  |  {bb:8.4f}")

Rezim : [2, 1, 2]  (dokumen: Bearish, Bullish, Bearish)
  0.0000    0.0000    0.0000    1.0000   -0.0198    0.0100  |    0.0152
  1.0000    0.0152   -0.0198    0.0000    0.0000    0.0000  |   -0.0050
  0.0000    0.0000    0.0000    1.0000   -0.0050    0.0152  |    0.0200


---
# Tugas (ii) — Investigasi Isu Numerik pada Matriks $A$

Operasi dasar yang dipakai, semuanya ditulis sendiri:

In [7]:
def transpose(M):
    return [[M[i][j] for i in range(len(M))] for j in range(len(M[0]))]


def matvec(M, x):
    return [sum(M[i][j] * x[j] for j in range(len(x))) for i in range(len(M))]


def gram(M):
    baris, kol = len(M), len(M[0])
    G = [[0.0] * kol for _ in range(kol)]
    for j in range(kol):
        for k in range(kol):
            G[j][k] = sum(M[i][j] * M[i][k] for i in range(baris))
    return G


def kolom(M, j):
    return [M[i][j] for i in range(len(M))]


def dot(u, v):
    return sum(a * c for a, c in zip(u, v))


def norm2(v):
    return sqrt(sum(x * x for x in v))

### (a) Beda skala antarkolom

Kolom indikator berisi 0/1, kolom lag berorde $10^{-2}$.

In [8]:
norma = [norm2(kolom(A, j)) for j in range(n)]

for j in range(n):
    print(f"  {NAMA_KOLOM[j]:<15} {norma[j]:10.4f}")
print(f"\nRasio terbesar/terkecil = {max(norma) / min(norma):.4f}")

  I                  13.9642
  I*R(t-1)            0.1297
  I*R(t-2)            0.1382
  1-I                10.2470
  (1-I)*R(t-1)        0.1010
  (1-I)*R(t-2)        0.0890

Rasio terbesar/terkecil = 156.9436


### (b) Kolinearitas antarkolom

$$c_{jk} = \frac{a_j^{T} a_k}{\lVert a_j\rVert_2\, \lVert a_k \rVert_2}$$

$|c_{jk}|$ mendekati 1 berarti dua kolom hampir sejajar.

In [9]:
kos = [[dot(kolom(A, j), kolom(A, k)) / (norma[j] * norma[k])
        for k in range(n)] for j in range(n)]

for j in range(n):
    print(f"  {NAMA_KOLOM[j]:<15}" + " ".join(f"{kos[j][k]:7.4f}"
                                              for k in range(n)))
print(f"\nTerbesar antara dua kolom berbeda = {kos[0][1]:.4f}")

  I               1.0000  0.7979  0.3237  0.0000  0.0000  0.0000
  I*R(t-1)        0.7979  1.0000  0.1575  0.0000  0.0000  0.0000
  I*R(t-2)        0.3237  0.1575  1.0000  0.0000  0.0000  0.0000
  1-I             0.0000  0.0000  0.0000  1.0000 -0.6809  0.1321
  (1-I)*R(t-1)    0.0000  0.0000  0.0000 -0.6809  1.0000 -0.0091
  (1-I)*R(t-2)    0.0000  0.0000  0.0000  0.1321 -0.0091  1.0000

Terbesar antara dua kolom berbeda = 0.7979


### (c) Struktur blok $A^{T}A$

Hasil kali kolom antarrezim selalu memuat faktor $I_t(1-I_t) = 0$,
jadi blok silangnya nol.

In [10]:
G = gram(A)

for i in range(n):
    print("  " + " ".join(f"{G[i][j]:11.4e}" for j in range(n)))
silang = max(abs(G[i][j]) for i in range(3) for j in range(3, 6))
print(f"\nElemen terbesar blok silang = {silang:.1f}")

   1.9500e+02  1.4456e+00  6.2479e-01  0.0000e+00  0.0000e+00  0.0000e+00
   1.4456e+00  1.6832e-02  2.8251e-03  0.0000e+00  0.0000e+00  0.0000e+00
   6.2479e-01  2.8251e-03  1.9111e-02  0.0000e+00  0.0000e+00  0.0000e+00
   0.0000e+00  0.0000e+00  0.0000e+00  1.0500e+02 -7.0440e-01  1.2047e-01
   0.0000e+00  0.0000e+00  0.0000e+00 -7.0440e-01  1.0192e-02 -8.1961e-05
   0.0000e+00  0.0000e+00  0.0000e+00  1.2047e-01 -8.1961e-05  7.9168e-03

Elemen terbesar blok silang = 0.0


### Rangkuman isu dan mitigasi

| Isu | Mitigasi |
|---|---|
| Harga punya tren naik kuat, kolom lag bisa hampir sejajar | Pemodelan dilakukan pada deret *return*, bukan harga mentah |
| Beda skala antarkolom besar (rasio 156,94) | Kolom bisa diskalakan dulu, atau pakai QR yang tidak membentuk $A^{T}A$ |
| Pembentukan $A^{T}A$ menguadratkan *condition number* | Dipakai Cholesky yang otomatis gagal bila $A^{T}A$ tidak lagi definit positif |
| Rezim ditentukan tanda $R_{t-1}$, nilai dekat nol rawan salah rezim | Diperiksa tidak ada *return* yang terlalu dekat nol dibanding galat pembulatan |

---
# Tugas (iii) — Penyelesaian via Persamaan Normal

Meminimumkan $\lVert A\vec{x}-\vec{b}\rVert_2^2$ memberi persamaan normal

$$A^{T}A\,\vec{x} = A^{T}\vec{b}$$

$A^{T}A$ simetrik definit positif, jadi dipakai Cholesky
$A^{T}A = LL^{T}$ lalu substitusi maju dan mundur.

In [11]:
def cholesky(G):
    N = len(G)
    L = [[0.0] * N for _ in range(N)]
    for i in range(N):
        for j in range(i + 1):
            s = G[i][j] - sum(L[i][k] * L[j][k] for k in range(j))
            if i == j:
                if s <= 0:
                    raise ValueError("G tidak definit positif")
                L[i][i] = sqrt(s)
            else:
                L[i][j] = s / L[j][j]
    return L


def cholesky_solve(L, rhs):
    N = len(L)
    y = [0.0] * N
    for i in range(N):
        # substitusi maju: L y = rhs
        y[i] = (rhs[i] - sum(L[i][j] * y[j] for j in range(i))) / L[i][i]
    x = [0.0] * N
    for i in range(N - 1, -1, -1):
        # substitusi mundur: L^T x = y
        x[i] = (y[i] - sum(L[j][i] * x[j] for j in range(i + 1, N))) / L[i][i]
    return x


c = matvec(transpose(A), b)
x = cholesky_solve(cholesky(G), c)

for j in range(n):
    print(f"  {NAMA_PARAM[j]:<10} = {x[j]:+.9f}")

r = [dot(A[i], x) - b[i] for i in range(m)]
print(f"\nNorma residual ||A x - b|| = {norm2(r):.8f}")

  alpha_1    = +0.001513374
  phi_1,1    = +0.172152897
  phi_1,2    = +0.166072851
  alpha_2    = -0.001212689
  phi_2,1    = -0.360376824
  phi_2,2    = -0.109808039

Norma residual ||A x - b|| = 0.15297705


### Condition number

Definisi yang dipakai di kelas,

$$\kappa_2(M) = \lVert M \rVert_2 \cdot \lVert M^{-1} \rVert_2$$

tidak bisa langsung dipakai pada $A$, karena $A$ berukuran
$300 \times 6$ sehingga $A^{-1}$ tidak ada.

Tetapi $A^{T}A$ berukuran $6\times6$, simetrik, dan definit positif.
Untuk matriks seperti itu berlaku
$\lVert A^{T}A \rVert_2 = \lambda_{\max}$ dan
$\lVert (A^{T}A)^{-1} \rVert_2 = 1/\lambda_{\min}$, sehingga

$$\kappa_2(A^{T}A) = \lambda_{\max} \cdot \frac{1}{\lambda_{\min}}
= \frac{\lambda_{\max}}{\lambda_{\min}},
\qquad \kappa_2(A) = \sqrt{\kappa_2(A^{T}A)}$$

Jadi tinggal mencari nilai eigen terbesar dan terkecil dari $A^{T}A$.
Keduanya dicari dengan **iterasi pangkat**:

- $\lambda_{\max}$: kalikan vektor dengan $A^{T}A$ berulang kali lalu
  normalkan. Vektornya menuju arah eigen dominan.
- $\lambda_{\min}$: lakukan hal yang sama tetapi dengan
  *menyelesaikan* $A^{T}A\,w = v$ (memakai Cholesky yang sudah ada),
  yang setara mengalikan dengan $(A^{T}A)^{-1}$, sehingga yang dominan
  justru eigen terkecil.

Nilai eigennya dihitung dengan $\lambda = v^{T}(A^{T}A)v$ untuk $v$
yang sudah dinormalkan.

In [12]:
def iterasi_pangkat(M, v, kali, putaran):
    """Cari arah eigen dominan dari operasi `kali`, lalu kembalikan
    nilai eigennya lewat v^T M v."""
    for _ in range(putaran):
        w = kali(v)
        panjang = norm2(w)
        v = [t / panjang for t in w]
    return dot(v, matvec(M, v)), v


L = cholesky(G)
awal = [1.0] * n

# lambda_max: kalikan berulang dengan G
lam_maks, _ = iterasi_pangkat(G, awal, lambda v: matvec(G, v), 500)

# lambda_min: selesaikan G w = v berulang (setara kali G^-1)
lam_min, _ = iterasi_pangkat(G, awal, lambda v: cholesky_solve(L, v), 3000)

kappa_G = lam_maks / lam_min
kappa_A = sqrt(kappa_G)

print(f"lambda_maks    = {lam_maks:.6e}")
print(f"lambda_min     = {lam_min:.6e}")
print()
print(f"kappa_2(A)     = {kappa_A:12.4f}")
print(f"kappa_2(A^T A) = {kappa_G:12.4f}")
print(f"kappa_2(A)^2   = {kappa_A ** 2:12.4f}")

lambda_maks    = 1.950127e+02
lambda_min     = 5.256855e-03

kappa_2(A)     =     192.6054
kappa_2(A^T A) =   37096.8440
kappa_2(A)^2   =   37096.8440


### Pengaruh terhadap stabilitas dan presisi

$\kappa_2(A^{T}A) = \kappa_2(A)^2$ secara persis: membentuk $A^{T}A$
**menguadratkan** *condition number*. Sistem dengan *condition number*
$\kappa$ kehilangan sekitar $\log_{10}\kappa$ angka desimal.

In [13]:
print(f"Lewat A langsung       : hilang ~{log10(kappa_A):.1f} angka")
print(f"Lewat persamaan normal : hilang ~{log10(kappa_G):.1f} angka")
print(f"Tersisa dari ~16 angka : ~{16 - log10(kappa_G):.1f} angka benar")

Lewat A langsung       : hilang ~2.3 angka
Lewat persamaan normal : hilang ~4.6 angka
Tersisa dari ~16 angka : ~11.4 angka benar


Untuk data ini persamaan normal masih aman. Tetapi bila $\kappa_2(A)$
jauh lebih besar, $\kappa_2(A)^2$ bisa melewati batas presisi mesin dan
solusinya tidak dapat dipercaya. Di situ QR lebih aman karena bekerja
langsung pada $A$ tanpa membentuk $A^{T}A$.